In [ ]:
import sys
import os
import numpy as np
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.utils import read_image_rgb, show_image

import matplotlib.pyplot as plt
%matplotlib inline

print("Setup complete. Project root set to:", project_root)

In [ ]:
input_path = os.path.join(project_root, 'data', 'input', 'test.jpg')

try:
    original_image = read_image_rgb(input_path)
    print(f"Image loaded successfully. Shape: {original_image.shape}")

except Exception as e:
    print(f"Failed to load image: {e}")

In [ ]:
import cv2

def compute_energy_map(img):
    """
    Computes the energy map of an image using the Dual-Gradient Energy function.
    
    The energy of a pixel is defined as E(p) = |dI/dx| + |dI/dy|.
    We use the Sobel operator to approximate the gradients.
    
    Args:
        img: Input image (RGB numpy array).
    
    Returns:
        energy_map: 2D numpy array (float64) of the same height and width as input.
    """

    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    
    dx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    dy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    
    energy_map = np.abs(dx) + np.abs(dy)
    
    return energy_map

energy_map = compute_energy_map(original_image)

In [ ]:
fig, (ax_original, ax_energy) = plt.subplots(1, 2, figsize=(15, 8))

# Original
ax_original.imshow(original_image)
ax_original.set_title("Original Image")
ax_original.axis('off')

# Energy 
energy_heatmap = ax_energy.imshow(energy_map, cmap='inferno')
ax_energy.set_title("Energy Map (Gradient Magnitude)")
ax_energy.axis('off')

fig.colorbar(energy_heatmap, ax=ax_energy, fraction=0.046, pad=0.04)

plt.show()

In [ ]:
from numba import jit

@jit(nopython=True)
def compute_cumulative_map(energy):
    """
    Computes the cumulative minimum energy map using Dynamic Programming.
    
    Args:
        energy: The energy map.
        
    Returns:
        M: The cumulative cost matrix.
    """
    rows, cols = energy.shape
    
    M = energy.copy()

    for r in range(1, rows):
        for c in range(cols):
            
            if c == 0:
                min_prev = min(M[r-1, c], M[r-1, c+1])
            
            elif c == cols - 1:
                min_prev = min(M[r-1, c-1], M[r-1, c])
            
            else:
                min_prev = min(M[r-1, c-1], M[r-1, c], M[r-1, c+1])
            
            M[r, c] += min_prev
            
    return M

cost_matrix = compute_cumulative_map(energy_map)
print(f"Cumulative Map computed. Shape: {cost_matrix.shape}")

In [ ]:
plt.figure(figsize=(10, 8))

plt.imshow(cost_matrix, cmap='viridis')
# Lighter = Higher Cumulative Cost  
plt.title("Cumulative Cost Matrix (M)")
plt.colorbar(label="Total Path Cost")
plt.axis('off')
plt.show()

In [ ]:
def find_vertical_seam(cost_matrix):
    """
    Finds the path of the seam with the lowest total cost using backtracking.
    
    Starting from the minimum value in the last row, we move upward, 
    always choosing the neighbor with the minimum cumulative energy.
    
    Args:
        cost_matrix: The cumulative cost matrix.
        
    Returns:
        seam: A 1D numpy array of shape (H,) containing the column index for each row.
              seam[y] = x means at row y, the pixel to remove is at column x.
    """
    rows, cols = cost_matrix.shape
    
    seam = np.zeros(rows, dtype=np.int32)
    
    current_col = np.argmin(cost_matrix[-1])
    seam[-1] = current_col
    
    for r in range(rows - 2, -1, -1):
        prev_c = current_col
        
        start_c = max(0, prev_c - 1)
        end_c = min(cols, prev_c + 2)
        
        neighbors = cost_matrix[r, start_c:end_c]
        
        offset = np.argmin(neighbors)
        current_col = start_c + offset
        
        seam[r] = current_col
        
    return seam

seam = find_vertical_seam(cost_matrix)
print(f"Seam found. Length: {len(seam)}")
print(f"Seam starts at col {seam[0]} and ends at col {seam[-1]}")

In [ ]:
img_with_seam = original_image.copy()

for r in range(img_with_seam.shape[0]):
    c = seam[r]
    
    if 0 <= c < img_with_seam.shape[1]:
        img_with_seam[r, c] = [255, 0, 0]

plt.figure(figsize=(10, 8))
plt.imshow(img_with_seam)
plt.title("Original Image with Optimal Seam (Red)")
plt.axis('off')
plt.show()

In [ ]:
def remove_vertical_seam(img, seam):
    """
    Removes the vertical seam using vectorized boolean masking.
    
    Args:
        img: Input image (H, W, 3).
        seam: Seam path (H,) containing column indices to remove.
        
    Returns:
        output: New image with shape (H, W-1, 3).
    """
    rows, cols, channels = img.shape
    
    mask = np.ones((rows, cols), dtype=bool)
    
    mask[np.arange(rows), seam] = False

    output = img[mask].reshape(rows, cols - 1, channels)
    
    return output

print(f"Original shape: {original_image.shape}")
carved_image = remove_vertical_seam(original_image, seam)
print(f"Carved shape:   {carved_image.shape}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 8))

ax1.imshow(original_image)
ax1.set_title(f"Original: {original_image.shape[1]}px width")
ax1.axis('off')

ax2.imshow(carved_image)
ax2.set_title(f"Carved: {carved_image.shape[1]}px width")
ax2.axis('off')

plt.show()